# ED token distribution: training labels, validation labels and predictions

This notebook answers three questions for one trained ED model:

1. Do the **training** and **validation** labels use tokens with similar frequencies?
2. Does the model **predict** tokens with the same frequencies as the validation labels?
3. Which tokens does the model predict **too often** or **too rarely**?

**How tokens are counted**

- *Labels* are encoded with the run's own frozen tokenizer.
- *Predictions* are the token IDs the model actually emits on the validation set: the best-scoring token in every output frame, repeats merged, blanks removed (TVA's `BestPath`). The predicted text is **not** re-encoded, because re-encoding could choose a different split than the model did.

Only inference is run; nothing is trained. You need the extracted ED data (`data/tva/ed/gold_wd`, `gold_wi`) and the saved run with its checkpoint under `results/thesis/ed/`. A GPU is used if available; CPU also works (slower).

In [ ]:
%matplotlib inline
from collections import Counter

import jiwer
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

from loguru import logger
from tva.ed_analysis import (CONDITIONS, collapse, js_divergence, load_records, load_run,
                             predict_frames, saved_predictions, show, show_table, token_counts)

logger.disable('tva')  # hide routine 'tokenizer loaded' messages

# ---- Settings: change these, then run all cells ----
CONDITION = 'handwriting_bigram'   # any name printed below
SPLIT = 'wd'                       # 'wd' or 'wi'
FOLD = 0                           # fold of a completed run
CHECKPOINT = 'best_cer.pth'        # or 'best_wer.pth', 'latest.pth'
TOP_N = 25                         # tokens shown in the bar chart
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Conditions:', ', '.join(CONDITIONS))

In [ ]:
def analyse(condition, split, fold):
    """Encode labels and run the model once; return everything later cells need."""
    run = load_run(condition, split, fold, CHECKPOINT, DEVICE)
    tokenizer = run['tokenizer']
    train = load_records(split, fold, 'train')
    references = [record['label'] for record in run['dataset'].annos]
    predicted_ids = [collapse(frames) for frames in predict_frames(run)]
    predicted_text = [tokenizer.decode(ids) for ids in predicted_ids]
    return {
        'run': run, 'tokenizer': tokenizer, 'references': references,
        'predicted_text': predicted_text,
        'counts': {
            'Training labels': token_counts(tokenizer, (tokenizer.encode(r['label']) for r in train)),
            'Validation labels': token_counts(tokenizer, (tokenizer.encode(t) for t in references)),
            'Predictions': token_counts(tokenizer, predicted_ids),
        },
        'sentences': {'Training labels': len(train), 'Validation labels': len(references),
                      'Predictions': len(references)},
        'characters': {'Training labels': sum(len(r['label']) for r in train),
                       'Validation labels': sum(map(len, references)),
                       'Predictions': sum(map(len, predicted_text))},
        'cer': jiwer.cer(references, predicted_text),
    }


result = analyse(CONDITION, SPLIT, FOLD)
run, tokenizer, counts = result['run'], result['tokenizer'], result['counts']
print(f"{CONDITION} | {SPLIT.upper()} fold {FOLD} | {CHECKPOINT} from epoch {run['epoch']} | {DEVICE}")
print(f"{result['sentences']['Training labels']} training and {result['sentences']['Validation labels']} validation sentences;",
      f'{tokenizer.size} classes including blank')
print(f"Validation CER of these predictions: {100 * result['cer']:.2f}%")
saved = saved_predictions(run)
if saved:
    same = sum(a == b for a, b in zip(saved, result['predicted_text']))
    print(f'Identical to the prediction saved during training for {same} of {len(saved)} sentences '
          '(small differences come from batching/mixed precision).')

## 1. Overview

- **Tokens per sentence** — how long the label sequences are. If predictions have *fewer* tokens than validation labels, the model drops tokens (deletions); *more* means insertions.
- **Characters per token** — how much a tokenizer compresses text (1.00 = plain characters).
- **Multi-character tokens** — how often learned pieces (bigrams, BPE/Unigram pieces) are used instead of single characters.
- **Different tokens used** — how much of the vocabulary actually appears (blank excluded).
- **Jensen–Shannon divergence** compares two frequency distributions: **0 = identical**, **1 = no tokens in common**.

In [ ]:
def overview(result):
    rows = []
    for name, c in result['counts'].items():
        total = sum(c.values())
        rows.append({
            'Source': name,
            'Tokens': f'{total:,}',
            'Tokens per sentence': f"{total / result['sentences'][name]:.1f}",
            'Characters per token': f"{result['characters'][name] / total:.2f}" if total else '-',
            'Multi-character tokens (%)': f'{100 * sum(n for t, n in c.items() if len(t) > 1) / total:.1f}' if total else '-',
            'Different tokens used': f"{len(c)} of {result['tokenizer'].size - 1}",
        })
    return rows


show_table(overview(result))
jsd_train_val = js_divergence(counts['Training labels'], counts['Validation labels'])
jsd_val_pred = js_divergence(counts['Validation labels'], counts['Predictions'])
show_table([
    {'Comparison': 'Training labels vs validation labels', 'Jensen–Shannon divergence': f'{jsd_train_val:.4f}'},
    {'Comparison': 'Validation labels vs predictions', 'Jensen–Shannon divergence': f'{jsd_val_pred:.4f}'},
])

## 2. Most frequent tokens

Each bar is a token's share of all tokens in that source. Tokens are ranked by training frequency; spaces are shown inside quotes, e.g. `'e '`.

If the three bars of a token have similar length, the model uses that token about as often as the labels do.

In [ ]:
top = [token for token, _ in counts['Training labels'].most_common(TOP_N)]
fig, ax = plt.subplots(figsize=(8, 0.33 * len(top) + 1.5))
for k, (name, c) in enumerate(counts.items()):
    total = sum(c.values()) or 1
    ax.barh([i + (k - 1) * 0.27 for i in range(len(top))], [100 * c[t] / total for t in top],
            height=0.27, label=name)
ax.set_yticks(range(len(top)), [show(t) for t in top])
ax.invert_yaxis()
ax.set_xlabel('Share of all tokens in that source (%)')
ax.set_title(f'Top {len(top)} training tokens — {CONDITION}, {SPLIT.upper()} fold {FOLD}')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Tokens predicted too often or too rarely

**Ratio = times predicted ÷ times in the validation labels.**

- about **1** — the model predicts the token about as often as it should;
- **below 1** — under-predicted: the model misses it or writes those characters another way (for example as single characters);
- **above 1** — over-predicted.

Only tokens that occur at least `MIN_COUNT` times in the validation labels are ranked, so rare tokens do not dominate.

In [ ]:
MIN_COUNT = 20
reference, predicted = counts['Validation labels'], counts['Predictions']
rows = [{'Token': show(t), 'Characters': len(t), 'In validation labels': reference[t],
         'Predicted': predicted[t], 'Ratio': round(predicted[t] / reference[t], 2)}
        for t in reference if reference[t] >= MIN_COUNT]
under = sorted((r for r in rows if r['Ratio'] < 1), key=lambda r: (r['Ratio'], r['Token']))
over = sorted((r for r in rows if r['Ratio'] > 1), key=lambda r: (-r['Ratio'], r['Token']))
display(Markdown('**Most under-predicted tokens** (ratio below 1)'))
show_table(under[:10])
display(Markdown('**Most over-predicted tokens** (ratio above 1)'))
show_table(over[:10])

never = sorted((t for t in reference if predicted[t] == 0), key=lambda t: (-reference[t], t))
extra = sorted((t for t in predicted if reference[t] == 0), key=lambda t: (-predicted[t], t))
print(f'{len(never)} tokens occur in the validation labels but are never predicted:',
      ', '.join(show(t) for t in never[:30]) or '-')
print(f'{len(extra)} tokens are predicted but never occur in the validation labels:',
      ', '.join(show(t) for t in extra[:30]) or '-')

## 4. Single characters versus learned pieces

Share of tokens by token length. If predictions contain fewer long tokens than the validation labels, the model tends to fall back on single characters. For the character tokenizer every token has length 1.

In [ ]:
lengths = sorted({len(t) for c in counts.values() for t in c})
rows = []
for name, c in counts.items():
    total = sum(c.values()) or 1
    row = {'Source': name}
    row.update({f'{n} character{"s" if n > 1 else ""} (%)': f'{100 * sum(v for t, v in c.items() if len(t) == n) / total:.1f}'
                for n in lengths})
    rows.append(row)
show_table(rows)

## 5. Summary in words

Generated from the numbers above. The divergence thresholds (below 0.02 = very similar, below 0.10 = moderately different) are rules of thumb for reading, not statistical tests.

In [ ]:
def similarity(value):
    if value < 0.02:
        return 'very similar'
    return 'moderately different' if value < 0.10 else 'clearly different'


def per_sentence(name):
    return sum(counts[name].values()) / result['sentences'][name]


def multi_share(name):
    total = sum(counts[name].values()) or 1
    return 100 * sum(n for t, n in counts[name].items() if len(t) > 1) / total


change = 100 * (per_sentence('Predictions') / per_sentence('Validation labels') - 1)
lines = [
    f'- Training and validation token frequencies are **{similarity(jsd_train_val)}** '
    f'(divergence {jsd_train_val:.3f}), so the fold split itself '
    f'{"barely changes" if jsd_train_val < 0.02 else "changes"} which tokens are needed.',
    f'- The model\'s predictions are **{similarity(jsd_val_pred)}** from the validation labels '
    f'(divergence {jsd_val_pred:.3f}).',
    f'- Predictions contain **{abs(change):.1f}% {"fewer" if change < 0 else "more"} tokens** per sentence than the '
    f'validation labels ({per_sentence("Predictions"):.1f} vs {per_sentence("Validation labels"):.1f}).',
    f'- Multi-character tokens make up **{multi_share("Predictions"):.1f}%** of predicted tokens versus '
    f'**{multi_share("Validation labels"):.1f}%** of validation-label tokens.',
    f'- **{len(never)}** tokens needed by the validation labels are never predicted.',
]
display(Markdown('\n'.join(lines)))

## 6. Optional: compare several tokenizers

Set `RUN_COMPARISON = True` to repeat the analysis for each condition in `COMPARE` (same split and fold). Conditions without a saved run are skipped. Each condition runs the model once on the validation set.

In [ ]:
RUN_COMPARISON = False
COMPARE = ['character', 'handwriting_bigram', 'linguistic_bigram_iam_frequency',
           'handwriting_bpe', 'linguistic_bpe', 'handwriting_unigram', 'linguistic_unigram']

if RUN_COMPARISON:
    rows = []
    for condition in COMPARE:
        try:
            other = analyse(condition, SPLIT, FOLD)
        except FileNotFoundError as error:
            print('Skipped:', error)
            continue
        labels, predictions = (dict(row) for row in overview(other)[1:])
        rows.append({
            'Condition': condition,
            'Tokens per sentence (labels → predicted)': f"{labels['Tokens per sentence']} → {predictions['Tokens per sentence']}",
            'Multi-character % (labels → predicted)': f"{labels['Multi-character tokens (%)']} → {predictions['Multi-character tokens (%)']}",
            'Divergence labels vs predictions': f"{js_divergence(other['counts']['Validation labels'], other['counts']['Predictions']):.4f}",
            'CER (%)': f"{100 * other['cer']:.2f}",
        })
        del other
    show_table(rows)